# Build Snowflake Feature Store with getML

## Prerequisites

```bash
mise env --dotenv > notebooks/.env
```

In [1]:
import getml
from dotenv import load_dotenv

PROJECT_NAME = "snowflake_feature_store"

load_dotenv(dotenv_path=".env")
getml.set_project(PROJECT_NAME)

  Loading pipelines...                     ━━━━━━━━━━━━━━━ 100% • 00:00


Connected to project 'snowflake_feature_store'.

## Setup and Data Loading 

In [2]:
import os
from snowflake.snowpark import Session

connection_params: dict[str, str | int] = {
    "account": os.environ["SNOWFLAKE_ACCOUNT"],
    "user": os.environ["SNOWFLAKE_USER"],
    "password": os.environ["SNOWFLAKE_PASSWORD"],
    "role": os.environ["SNOWFLAKE_ROLE"],
    "warehouse": os.environ["SNOWFLAKE_WAREHOUSE"],
    "database": os.environ["SNOWFLAKE_DATABASE"],
    "schema": os.environ["SNOWFLAKE_SCHEMA"],
}

session = Session.builder.configs(connection_params).create()

In [3]:
weekly_sales_by_store = session.table(
    "PREPARED.WEEKLY_SALES_BY_STORE_WITH_TARGET"
).to_arrow()
weekly_sales_by_store = getml.DataFrame.from_arrow(
    weekly_sales_by_store, name="weekly_sales_by_store"
)

orders = session.table("RAW.RAW_ORDERS").to_arrow()
orders = getml.DataFrame.from_arrow(orders, name="orders")

DataFrame.to_arrow() is experimental since 1.28.0. Do not use it in production. 
/home/alex/projects/worktrees/getml-demo/60-create-initial-snowflake-notebook-5-sections/integration/snowflake/notebooks/.venv/lib/python3.12/site-packages/getml/data/_io/arrow.py:371: UserWarning:     
Column 'NEXT_WEEK_SALES' has been converted from decimal to float. This may
    result in a loss of precision!
    
  warnings.warn(


## getml Annotations

In [4]:
weekly_sales_by_store.set_role(
    cols=["STORE_ID", "SNAPSHOT_ID"], role=getml.data.roles.join_key
)
weekly_sales_by_store.set_role(cols="REFERENCE_DATE", role=getml.data.roles.time_stamp)
weekly_sales_by_store.set_role(cols="NEXT_WEEK_SALES", role=getml.data.roles.target)
weekly_sales_by_store.set_role(
    cols=[
        "STORE_NAME",
        "YEAR",
        "MONTH",
        "WEEK_NUMBER",
        "IS_FULL_WEEK_AFTER_OPENING",
        "HAS_ORDER_ACTIVITY",
        "HAS_MIN_HISTORY",
    ],
    role=getml.data.roles.categorical,
)
weekly_sales_by_store.set_role(
    cols=["DAYS_SINCE_OPEN", "NEXT_WEEK_ORDERS"], role=getml.data.roles.numerical
)


In [5]:
orders.set_role(cols=["STORE_ID", "ID", "CUSTOMER"], role=getml.data.roles.join_key)
orders.set_role(
    cols="ORDERED_AT",
    role=getml.data.roles.time_stamp,
    time_formats=["%Y-%m-%dT%H:%M:%S"],
)
orders.set_role(
    cols=["SUBTOTAL", "ORDER_TOTAL", "TAX_PAID"], role=getml.data.roles.numerical
)

In [6]:
weekly_sales_by_store

name,REFERENCE_DATE,STORE_ID,SNAPSHOT_ID,NEXT_WEEK_SALES,STORE_NAME,YEAR,MONTH,WEEK_NUMBER,IS_FULL_WEEK_AFTER_OPENING,HAS_ORDER_ACTIVITY,HAS_MIN_HISTORY,DAYS_SINCE_OPEN,NEXT_WEEK_ORDERS
role,time_stamp,join_key,join_key,target,categorical,categorical,categorical,categorical,categorical,categorical,categorical,numerical,numerical
unit,"time stamp, comparison only",,,,,,,,,,,,
0,2019-05-20,fc7707c0-2f1e-48d4-b870-7cbeddfc...,48,13165.49,Philadelphia,2019,5,21,true,true,true,261,1126
1,2020-04-27,eafbd328-0434-4f46-9c2d-cc97a46f...,145,24888.19,Brooklyn,2020,4,18,true,true,true,412,2346
2,2020-04-06,eafbd328-0434-4f46-9c2d-cc97a46f...,139,25648.41,Brooklyn,2020,4,15,true,true,true,391,2386
3,2019-07-08,eafbd328-0434-4f46-9c2d-cc97a46f...,61,10507.1,Brooklyn,2019,7,28,true,true,true,118,1113
4,2018-12-03,fc7707c0-2f1e-48d4-b870-7cbeddfc...,14,8689.4,Philadelphia,2018,12,49,true,true,true,93,737
,...,...,...,...,...,...,...,...,...,...,...,...,...
1374,2023-12-11,eafbd328-0434-4f46-9c2d-cc97a46f...,1162,29879.13,Brooklyn,2023,12,50,true,true,true,1735,2753
1375,2024-05-27,61743c9b-2394-4d36-8062-8a6820fa...,1302,16341.42,Los Angeles,2024,5,22,true,true,true,988,1432


In [7]:
orders

name,ORDERED_AT,STORE_ID,ID,CUSTOMER,SUBTOTAL,ORDER_TOTAL,TAX_PAID
role,time_stamp,join_key,join_key,join_key,numerical,numerical,numerical
unit,"time stamp, comparison only",,,,,,
0,2020-05-18 07:44:00,fc7707c0-2f1e-48d4-b870-7cbeddfc...,1d78a778-9c83-4092-b185-3e0cb4c8...,2d54620c-28bd-44a4-aa0e-5031c01a...,600,636,36
1,2020-05-18 14:01:00,fc7707c0-2f1e-48d4-b870-7cbeddfc...,7d8ddf6a-c066-4121-a7fb-10572dcf...,0e0a7730-ba39-48ac-98fd-69293150...,400,424,24
2,2020-05-18 08:38:00,fc7707c0-2f1e-48d4-b870-7cbeddfc...,e664a36d-fcb6-4f6c-8473-b3264209...,d0ab73cc-8b20-4492-9bd9-ad922bea...,700,742,42
3,2020-05-18 14:43:00,fc7707c0-2f1e-48d4-b870-7cbeddfc...,cf5ee75a-68c4-45fe-872a-a7f5b005...,29e77d3b-14bf-4cd4-b4aa-54662ee2...,600,636,36
4,2020-05-18 17:00:00,fc7707c0-2f1e-48d4-b870-7cbeddfc...,0a73b088-99a4-424c-9ace-a9632fc5...,4c466e98-c2ea-438f-8be4-9669d9e1...,1600,1696,96
,...,...,...,...,...,...,...
2309598,2022-12-05 12:52:00,abfcc332-1eaf-42f6-b8e4-569bf6a3...,0bb3fbbf-bd42-4659-ae80-b28643cf...,be133bbe-beed-4e26-9844-bc742c3a...,400,425,25
2309599,2022-12-05 16:23:00,6964b061-b98d-43f2-9078-ef9c423e...,ea8bcf34-4d67-40e9-9568-3f20523c...,9b4479a5-0a03-4b2e-81b8-6e5df336...,2100,2257,157


## getML Data Model

In [8]:
validation_begin = getml.data.time.datetime(2023, 1, 1)
test_begin = getml.data.time.datetime(2024, 1, 1)

split = getml.data.split.time(
    population=weekly_sales_by_store,
    time_stamp="REFERENCE_DATE",
    validation=validation_begin,
    test=test_begin,
)

# Filter dataframes using the split column
weekly_sales_by_store_train = weekly_sales_by_store[split == "train"]
weekly_sales_by_store_validation = weekly_sales_by_store[split == "validation"]
weekly_sales_by_store_test = weekly_sales_by_store[split == "test"]

print(
    f"Training set size: {len(weekly_sales_by_store_train)}"
    f"\nValidation set size: {len(weekly_sales_by_store_validation)}"
    f"\nTest set size: {len(weekly_sales_by_store_test)}"
)

Training set size: 863
Validation set size: 312
Test set size: 204


In [9]:
weekly_sales_by_store_validation

name,REFERENCE_DATE,STORE_ID,SNAPSHOT_ID,NEXT_WEEK_SALES,STORE_NAME,YEAR,MONTH,WEEK_NUMBER,IS_FULL_WEEK_AFTER_OPENING,HAS_ORDER_ACTIVITY,HAS_MIN_HISTORY,DAYS_SINCE_OPEN,NEXT_WEEK_ORDERS
role,time_stamp,join_key,join_key,target,categorical,categorical,categorical,categorical,categorical,categorical,categorical,numerical,numerical
unit,"time stamp, comparison only",,,,,,,,,,,,
0,2023-02-27,abfcc332-1eaf-42f6-b8e4-569bf6a3...,914,24980.54,Chicago,2023,2,9,true,true,true,1035,2204
1,2023-05-15,fc7707c0-2f1e-48d4-b870-7cbeddfc...,983,18133.36,Philadelphia,2023,5,20,true,true,true,1717,1545
2,2023-05-22,abfcc332-1eaf-42f6-b8e4-569bf6a3...,986,23574.09,Chicago,2023,5,21,true,true,true,1119,2105
3,2023-08-28,fc7707c0-2f1e-48d4-b870-7cbeddfc...,1073,14535.01,Philadelphia,2023,8,35,true,true,true,1822,1260
4,2023-10-23,abfcc332-1eaf-42f6-b8e4-569bf6a3...,1118,26280.78,Chicago,2023,10,43,true,true,true,1273,2251
...,...,...,...,...,...,...,...,...,...,...,...,...,...


In [10]:
data_model = getml.data.DataModel(
    population=weekly_sales_by_store_train.to_placeholder("WEEKLY_SALES_BY_STORE")
)

# Add all peripheral tables
data_model.add(
    getml.data.to_placeholder(
        orders=orders,
    )
)

# Define relationships using joins
data_model.WEEKLY_SALES_BY_STORE.join(
    right=data_model.orders,
    on="STORE_ID",
    time_stamps=("REFERENCE_DATE", "ORDERED_AT"),
    relationship=getml.data.relationship.one_to_many,
    memory=getml.data.time.days(30),
)


In [11]:
container = getml.data.Container(
    train=weekly_sales_by_store_train,
    validation=weekly_sales_by_store_validation,
    test=weekly_sales_by_store_test,
)

# Add peripheral tables with aliases matching the data model placeholders
container.add(
    orders=orders,
)
container.save()
getml.project.data_frames.save()

## Training

In [12]:
fast_prop = getml.feature_learning.FastProp()

predictor = getml.predictors.XGBoostRegressor(
    n_jobs=0,
)

pipe = getml.Pipeline(
    data_model=data_model,
    feature_learners=[
        fast_prop,
    ],
    predictors=[predictor],
)

pipe.fit(container.train)

Checking data model...

  Staging...                               ━━━━━━━━━━━━━━━ 100% • 00:00
  Checking...                              ━━━━━━━━━━━━━━━ 100% • 00:01


The pipeline check generated 0 issues labeled INFO and 2 issues labeled WARNING.

To see the issues in full, run .check() on the pipeline.

  Staging...                               ━━━━━━━━━━━━━━━ 100% • 00:01
  FastProp: Trying 50 features...          ━━━━━━━━━━━━━━━ 100% • 00:00
  FastProp: Building features...           ━━━━━━━━━━━━━━━ 100% • 00:03
  XGBoost: Training as predictor...        ━━━━━━━━━━━━━━━ 100% • 00:02


Trained pipeline.

Time taken: 0:00:07.256700.



Pipeline(data_model='WEEKLY_SALES_BY_STORE',
         feature_learners=['FastProp'],
         feature_selectors=[],
         include_categorical=False,
         loss_function='SquareLoss',
         peripheral=['orders'],
         predictors=['XGBoostRegressor'],
         preprocessors=[],
         share_selected_features=0.5,
         tags=['container-tWR94s'])

In [13]:
predictions = pipe.predict(container.test)

# Calculate metrics
scores = pipe.score(container.test)
scores

  Staging...                               ━━━━━━━━━━━━━━━ 100% • 00:00
  Preprocessing...                         ━━━━━━━━━━━━━━━ 100% • 00:00
  FastProp: Building features...           ━━━━━━━━━━━━━━━ 100% • 00:03
  Staging...                               ━━━━━━━━━━━━━━━ 100% • 00:00
  Preprocessing...                         ━━━━━━━━━━━━━━━ 100% • 00:00
  FastProp: Building features...           ━━━━━━━━━━━━━━━ 100% • 00:03


,date time,set used,target,mae,rmse,rsquared
0,2025-12-16 23:45:26,train,NEXT_WEEK_SALES,249.1798,316.897,0.9973
1,2025-12-16 23:45:34,test,NEXT_WEEK_SALES,481.3215,629.0032,0.9856


## Generate getML Feature Interpretations

In [14]:
import getml_interpretations
from getml_interpretations import DomainReport

domain_report: DomainReport = await getml_interpretations.generate_domain_report(
    project_name=PROJECT_NAME,
    pipeline=pipe,
    container=container,
    model="gpt-5.2",
)

  Loading pipelines...                     ━━━━━━━━━━━━━━━ 100% • 00:00


Connected to project 'snowflake_feature_store'.

  Staging...                               ━━━━━━━━━━━━━━━ 100% • 00:00
  Preprocessing...                         ━━━━━━━━━━━━━━━ 100% • 00:00
  FastProp: Building features...           ━━━━━━━━━━━━━━━ 100% • 00:04
  Staging...                               ━━━━━━━━━━━━━━━ 100% • 00:00
  Preprocessing...                         ━━━━━━━━━━━━━━━ 100% • 00:00
  FastProp: Building features...           ━━━━━━━━━━━━━━━ 100% • 00:03
  Staging...                               ━━━━━━━━━━━━━━━ 100% • 00:00
  Preprocessing...                         ━━━━━━━━━━━━━━━ 100% • 00:00
  FastProp: Building features...           ━━━━━━━━━━━━━━━ 100% • 00:03
  Staging...                               ━━━━━━━━━━━━━━━ 100% • 00:00
  Preprocessing...                         ━━━━━━━━━━━━━━━ 100% • 00:00
  FastProp: Building features...           ━━━━━━━━━━━━━━━ 100% • 00:04
  Staging...                               ━━━━━━━━━━━━━━━ 100% • 00:00
  Preprocessing...                         ━━━━━━━━━━━━━━━ 100% 

In [15]:
domain_report.to_markdown()

In [16]:
from pathlib import Path

from getml_interpretations import ColumnDescriptionsReport

jaffle_shop_annotations = Path("annotations.yml")
user_prompt = f"""
Please use the following annotations as source of truth for generating the
column descriptions report:

{jaffle_shop_annotations.read_text()}
"""


column_descriptions_report: ColumnDescriptionsReport = (
    await getml_interpretations.generate_column_descriptions_report(
        project_name=PROJECT_NAME,
        pipeline=pipe,
        container=container,
        domain_report=domain_report,
        model="gpt-5.2",
        user_prompt=user_prompt,
    )
)

In [17]:
column_descriptions_report.to_json()

In [18]:
from getml_interpretations import FeatureDescriptionsReport

feature_descriptions_report: FeatureDescriptionsReport = (
    await getml_interpretations.generate_feature_descriptions_report(
        project_name=PROJECT_NAME,
        pipeline=pipe,
        container=container,
        domain_report=domain_report,
        column_descriptions_report=column_descriptions_report,
        model="gpt-5.2",
        batch_size=20,
    )
)

In [19]:
feature_descriptions_report.to_json()

## Feature Export

In [20]:
import pandas as pd

# Transform features using getML pipeline
features_np = pipe.transform(
    population_table=weekly_sales_by_store,
    peripheral_tables=[orders],
)

# Create DataFrame with meaningful feature names from getML pipeline
features_df = pd.DataFrame(features_np, columns=pipe.features.names)

# Add join keys for Snowflake Feature Store entity linking
features_df["STORE_ID"] = weekly_sales_by_store["STORE_ID"].to_numpy()
features_df["SNAPSHOT_ID"] = weekly_sales_by_store["SNAPSHOT_ID"].to_numpy()

# Uppercase column names to match Snowflake's unquoted identifier convention
features_df.columns = features_df.columns.str.upper()

# Write features to Snowflake table
session.write_pandas(
    df=features_df,
    table_name="GETML_FEATURES",
    schema="GETML_FS",
    auto_create_table=True,
    overwrite=True,
)

  Staging...                               ━━━━━━━━━━━━━━━ 100% • 00:00
  Preprocessing...                         ━━━━━━━━━━━━━━━ 100% • 00:00
  FastProp: Building features...           ━━━━━━━━━━━━━━━ 100% • 00:02


In [21]:
features_df

,FEATURE_1_1,FEATURE_1_2,FEATURE_1_3,FEATURE_1_4,FEATURE_1_5,FEATURE_1_6,FEATURE_1_7,FEATURE_1_8,FEATURE_1_9,FEATURE_1_10,...,FEATURE_1_45,FEATURE_1_46,FEATURE_1_47,FEATURE_1_48,FEATURE_1_49,FEATURE_1_50,DAYS_SINCE_OPEN,NEXT_WEEK_ORDERS,STORE_ID,SNAPSHOT_ID
0,1135.195054,80.0,4611.0,600.0,4300.0,9500.0,600.0,0.0,600.0,1279.079097,...,234000.0,708676.489468,6.034538e+09,0.0,539.309168,4691.0,261.0,1126.0,fc7707c0-2f1e-48d4-b870-7cbeddfc1aac,48
1,1058.410351,89.0,9649.0,1500.0,1800.0,9900.0,600.0,0.0,600.0,1114.206336,...,402720.0,708293.103091,1.269187e+10,0.0,259.903461,9738.0,412.0,2346.0,eafbd328-0434-4f46-9c2d-cc97a46faa13,145
2,1059.729512,84.0,9824.0,400.0,700.0,9900.0,600.0,0.0,600.0,1088.355072,...,1098000.0,707509.957824,1.282615e+10,0.0,255.401231,9908.0,391.0,2386.0,eafbd328-0434-4f46-9c2d-cc97a46faa13,139
3,1018.263736,77.0,4473.0,5100.0,1600.0,9700.0,600.0,0.0,600.0,1162.026676,...,406800.0,710144.524458,6.043895e+09,0.0,555.893603,4550.0,118.0,1113.0,eafbd328-0434-4f46-9c2d-cc97a46faa13,61
4,1169.625247,73.0,2462.0,500.0,1700.0,9300.0,600.0,0.0,600.0,1303.398856,...,406800.0,709068.101433,3.108209e+09,0.0,997.529597,2535.0,93.0,737.0,fc7707c0-2f1e-48d4-b870-7cbeddfc1aac,14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1374,1099.489302,88.0,11269.0,400.0,9200.0,9500.0,600.0,0.0,600.0,1156.497623,...,2048400.0,708333.099447,1.462573e+10,0.0,222.844311,11357.0,1735.0,2753.0,eafbd328-0434-4f46-9c2d-cc97a46faa13,1162
1375,1080.277167,78.0,5839.0,600.0,600.0,9600.0,600.0,0.0,600.0,1123.885161,...,2394000.0,711969.715803,7.678430e+09,0.0,427.778905,5917.0,988.0,1432.0,61743c9b-2394-4d36-8062-8a6820fa274a,1302
1376,1143.409056,83.0,6388.0,600.0,700.0,9700.0,600.0,0.0,600.0,1269.610589,...,320400.0,713615.753322,8.410696e+09,0.0,391.085008,6471.0,2109.0,1571.0,fc7707c0-2f1e-48d4-b870-7cbeddfc1aac,1319
1377,1080.887604,82.0,5799.0,1800.0,1100.0,9600.0,600.0,0.0,600.0,1118.385390,...,836400.0,709831.760379,7.672293e+09,0.0,430.295918,5881.0,1002.0,1418.0,61743c9b-2394-4d36-8062-8a6820fa274a,1314


In [22]:
from snowflake.ml.feature_store import CreationMode, Entity, FeatureStore, FeatureView


# Create Snowpark DataFrame from the table for Feature Store
features_snowpark_df = session.table("GETML_FS.GETML_FEATURES")

# Initialize Feature Store
snowflake_feature_store = FeatureStore(
    session=session,
    database=os.environ["SNOWFLAKE_DATABASE"],
    name="GETML_FS",
    default_warehouse=os.environ["SNOWFLAKE_WAREHOUSE"],
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
)

# Create Entity (required for FeatureView)
store_entity = Entity(name="STORE_SNAPSHOT", join_keys=["STORE_ID", "SNAPSHOT_ID"])
snowflake_feature_store.register_entity(store_entity)


/home/alex/projects/worktrees/getml-demo/60-create-initial-snowflake-notebook-5-sections/integration/snowflake/notebooks/.venv/lib/python3.12/site-packages/snowflake/ml/feature_store/feature_store.py:189: UserWarning: Entity STORE_SNAPSHOT already exists. Skip registration.
  return f(self, *args, **kargs)


Entity(name=STORE_SNAPSHOT, join_keys=['STORE_ID', 'SNAPSHOT_ID'], owner=None, desc=)

In [23]:
feature_dict = feature_descriptions_report.model_dump()
feature_dict.get("feature_descriptions")

{'feature_1_1': {'name': 'feature_1_1',
  'title': 'avg_subtotal_last_30d_by_store',
  'description': 'Average order subtotal [currency, pre-tax] for a store over the historical lookback window ending at the store-week reference timestamp.\n\n**How it’s computed (SQL logic)**\n- Join the weekly store-level table to the transactional `orders` table on `STORE_ID` (one-to-many).\n- Keep only transactions with `orders.ORDERED_AT <= weekly_sales_by_store.REFERENCE_DATE` (no future leakage relative to the reference week).\n- Apply a rolling lookback of ~30 days using the generated bound `ORDERED_AT__30_000000_days` (effectively `ORDERED_AT > REFERENCE_DATE - 30 days`).\n- Aggregate `AVG(orders.SUBTOTAL)` across the eligible orders.\n\n**Business meaning**\nRepresents recent average basket value before tax for the store, a proxy for price level / basket composition and recent customer spend behavior.\n\n**Statistical notes**\n- `SUBTOTAL` values are highly discretized (few price points overal

In [25]:
# Create external FeatureView (refresh_freq=None means externally managed)
weekly_sales_feature_view = FeatureView(
    name="weekly_sales_features",
    entities=[store_entity],
    feature_df=features_snowpark_df,
    refresh_freq=None,
    desc="Features generated by getML for weekly sales prediction",
)

# Build feature descriptions with uppercase keys to match Snowflake column names
# Extract description strings from nested dicts and uppercase the keys
feature_descs = feature_dict.get("feature_descriptions", {})
feature_descs_upper = {
    k.upper(): v.get("description", "") if isinstance(v, dict) else v
    for k, v in feature_descs.items()
}

weekly_sales_feature_view = weekly_sales_feature_view.attach_feature_desc(
    feature_descs_upper
)

# Register the FeatureView
registered_feature_view = snowflake_feature_store.register_feature_view(
    feature_view=weekly_sales_feature_view, version="1", overwrite=True
)

registered_feature_view

FeatureView(_name=WEEKLY_SALES_FEATURES, _entities=[Entity(name=STORE_SNAPSHOT, join_keys=['STORE_ID', 'SNAPSHOT_ID'], owner=None, desc=)], _feature_df=<snowflake.snowpark.dataframe.DataFrame object at 0x736191371c40>, _timestamp_col=None, _desc=Features generated by getML for weekly sales prediction, _infer_schema_df=<snowflake.snowpark.dataframe.DataFrame object at 0x7361912ff560>, _query=SELECT  *  FROM GETML_FS.GETML_FEATURES, _version=1, _status=FeatureViewStatus.STATIC, _feature_desc=OrderedDict({'FEATURE_1_1': 'Average order subtotal [currency, pre-tax] for a store over the historical lookback window ending at the store-week reference timestamp.\n\n**How it’s computed (SQL logic)**\n- Join the weekly store-level table to the transactional `orders` table on `STORE_ID` (one-to-many).\n- Keep only transactions with `orders.ORDERED_AT <= weekly_sales_by_store.REFERENCE_DATE` (no future leakage relative to the reference week).\n- Apply a rolling lookback of ~30 days using the generat

https://app.snowflake.com/pgciadt/sm09519/#/features/database/JAFFLE_SHOP/store/GETML_FS/feature-view/WEEKLY_SALES_FEATURES/version/1/feature-view-details